In [1]:
!pip install facenet-pytorch --no-deps


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 33.8 MB/s eta 0:00:00


In [2]:
import os
import shutil
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import torch
from facenet_pytorch import MTCNN

# -------------------------
# Config
# -------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

# ⚠️ CHANGE THIS TO YOUR FF++ FRAMES ROOT
SRC_ROOT = Path("/kaggle/input/datasets/hooriyamasood/faceforensics-c23/FaceForensics-c40_frames-Split")  
DST_ROOT = Path("/kaggle/working/FF_aligned_faces")

splits = ["train", "val", "test"]
manips = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]

# -------------------------
# Fresh output folder
# -------------------------
if DST_ROOT.exists():
    shutil.rmtree(DST_ROOT)
DST_ROOT.mkdir(parents=True, exist_ok=True)

# -------------------------
# MTCNN
# -------------------------
mtcnn = MTCNN(
    image_size=224,
    margin=40,
    device=device,
    post_process=False
)

# -------------------------
# Counters
# -------------------------
saved_count = 0
fallback_count = 0
fail_count = 0
fail_examples = []

# -------------------------
# Safe face save
# -------------------------
def save_aligned(src_img_path: Path, dst_img_path: Path):
    global saved_count, fallback_count, fail_count, fail_examples

    try:
        img = Image.open(src_img_path).convert("RGB")
        face = mtcnn(img)

        dst_img_path.parent.mkdir(parents=True, exist_ok=True)

        if face is None:
            img = img.resize((224, 224))
            img.save(dst_img_path)
            saved_count += 1
            fallback_count += 1
            return

        face = face.permute(1, 2, 0).contiguous()

        if face.dtype != torch.uint8:
            if float(face.max()) <= 1.5:
                face = face * 255.0
            face = face.clamp(0, 255).byte()

        face_np = face.detach().cpu().numpy()
        Image.fromarray(face_np).save(dst_img_path)
        saved_count += 1

    except Exception as e:
        fail_count += 1
        if len(fail_examples) < 20:
            fail_examples.append((str(src_img_path), repr(e)))

# -------------------------
# ALIGNMENT LOOP
# -------------------------

for split in splits:

    # ===== REAL =====
    real_src = SRC_ROOT / split / "real"
    real_dst = DST_ROOT / split / "real"

    if not real_src.exists():
        print(f"Missing real folder: {real_src}")
    else:
        for video_dir in tqdm(sorted(real_src.iterdir()), desc=f"{split}/real"):
            if not video_dir.is_dir():
                continue

            for img_path in sorted(video_dir.glob("*")):
                if img_path.is_file():
                    dst_path = real_dst / video_dir.name / img_path.name
                    save_aligned(img_path, dst_path)

    # ===== FAKE =====
    fake_src = SRC_ROOT / split / "fake"
    fake_dst = DST_ROOT / split / "fake"

    if not fake_src.exists():
        print(f"Missing fake folder: {fake_src}")
    else:
        for manip in manips:
            manip_src = fake_src / manip
            manip_dst = fake_dst / manip

            if not manip_src.exists():
                print(f"Missing manip folder: {manip_src}")
                continue

            for video_dir in tqdm(sorted(manip_src.iterdir()), desc=f"{split}/{manip}"):
                if not video_dir.is_dir():
                    continue

                for img_path in sorted(video_dir.glob("*")):
                    if img_path.is_file():
                        dst_path = manip_dst / video_dir.name / img_path.name
                        save_aligned(img_path, dst_path)

# -------------------------
# SUMMARY
# -------------------------
print("\nAlignment finished.")
print("Saved total   :", saved_count)
print("Fallback total:", fallback_count)
print("Failed total  :", fail_count)

if fail_examples:
    print("\nSample failures:")
    for p, err in fail_examples[:10]:
        print("FILE:", p)
        print("ERR :", err)
        print("-" * 80)

# -------------------------
# VERIFY (CORRECT WAY)
# -------------------------
print("\nVerifying output folders...")

for split in splits:
    # REAL count
    real_dir = DST_ROOT / split / "real"
    total_real = 0

    if real_dir.exists():
        for v in real_dir.iterdir():
            if v.is_dir():
                total_real += len(list(v.glob("*")))

    print(f"{split}/real images: {total_real}")

    # FAKE count
    fake_dir = DST_ROOT / split / "fake"
    total_fake = 0

    if fake_dir.exists():
        for manip in manips:
            manip_dir = fake_dir / manip
            if not manip_dir.exists():
                continue

            for v in manip_dir.iterdir():
                if v.is_dir():
                    total_fake += len(list(v.glob("*")))

    print(f"{split}/fake images: {total_fake}")

print("\nAligned dataset structure verified.")

test/NeuralTextures: 100%|██████████| 100/100 [01:15<00:00,  1.33it/s]



Alignment finished.
Saved total   : 50000
Fallback total: 7
Failed total  : 0

Verifying output folders...
train/real images: 8000
train/fake images: 32000
val/real images: 1000
val/fake images: 4000
test/real images: 1000
test/fake images: 4000

Aligned dataset structure verified.


In [3]:
!zip -r /kaggle/working/140k_aligned_faces.zip /kaggle/working/140k_aligned_faces

	zip warning: name not matched: /kaggle/working/140k_aligned_faces

zip error: Nothing to do! (try: zip -r /kaggle/working/140k_aligned_faces.zip . -i /kaggle/working/140k_aligned_faces)
